In [ ]:
# Step 1: Mount Drive
# ============================================================
# AGGREGATE BATCH CROPPING PIPELINE
# ============================================================
# Purpose : Process an entire folder of aggregate grid-sheet
# photographs and automatically crop each individual
# specimen into a separate image file.
# Input   : Folder of grid-sheet images on Google Drive
# Output  : Center-cropped 380 x 410 px images saved to Drive
# Pipeline: Adaptive threshold -> Morphological filter ->
# Hough lines -> Clustering -> Grid boxing -> Crop
# ============================================================

# Mount Google Drive to access the input folder and save results
from google.colab import drive
drive.mount('/content/drive')


# Core libraries -----------------------------------------------
# os        : File path construction and directory creation
# cv2       : OpenCV for image I/O, thresholding, morphology, Hough
# numpy     : Array operations for pixel-level manipulation
# matplotlib: Visualisation of intermediate and final results
# glob      : Pattern-based file discovery in input folder
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

# Step 2: Set input/output paths

# ── Paths ──────────────────────────────────────────────────────
# input_folder : Google Drive folder containing grid-sheet images
# output_folder: Destination for individual cropped specimens
# os.makedirs ensures the output folder exists before saving
input_folder = '/content/drive/MyDrive/folder_agg/folder_agg/0'
output_folder = '/content/drive/MyDrive/folder_agg/folder_agg/0_cropped'
os.makedirs(output_folder, exist_ok=True)

# Step 3: Process each image in the folder

# ── Main batch loop ────────────────────────────────────────────
# Discovers all image files in the input folder (sorted for
# reproducibility) and runs the full cropping pipeline on each.
image_paths = sorted(glob(os.path.join(input_folder, '*')))
print(f"Found {len(image_paths)} images")

for img_path in image_paths:
    print(f"\nProcessing: {os.path.basename(img_path)}")

    # ── Step 1: Load image and convert to grayscale ───────────────
    # Grayscale reduces the image to a 2-D intensity matrix,
    # sufficient for grid line detection and computationally cheaper
    # than operating on 3-channel colour data.
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape


    # ── Step 2: Adaptive thresholding ─────────────────────────────
    # Local (adaptive) thresholding handles non-uniform illumination
    # across the sheet. Block size 15 and constant 8 were tuned
    # empirically for this aggregate grid format.
    # THRESH_BINARY_INV makes grid lines appear white on black.
    thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV, 15, 8
    )


    # ── Step 3: Morphological filtering to isolate grid lines ─────
    # Horizontal kernel (50x1): preserves wide horizontal structures.
    # Vertical kernel   (1x50): preserves tall vertical structures.
    # MORPH_OPEN erodes then dilates, removing blobs smaller than
    # the kernel while retaining true grid lines intact.
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 1))
    vertical_kernel   = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 50))
    horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
    vertical   = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
    # Combine horizontal and vertical masks into a single grid skeleton
    grid_mask = cv2.add(horizontal, vertical)


    # ── Step 4: Probabilistic Hough Line Transform ────────────────
    # Detects line segments from the binary grid mask.
    # rho=1, theta=pi/180 : 1-pixel and 1-degree resolution.
    # threshold=100        : minimum accumulator votes for acceptance.
    # minLineLength=80     : rejects short fragments (texture noise).
    # maxLineGap=10        : bridges small gaps in collinear segments.
    lines = cv2.HoughLinesP(
        grid_mask, rho=1, theta=np.pi / 180,
        threshold=100, minLineLength=80, maxLineGap=10
    )


    # ── Step 5: Classify segments as horizontal or vertical ───────
    # Horizontal: near-zero vertical displacement   |y1-y2| < 10 px
    # Vertical  : near-zero horizontal displacement |x1-x2| < 10 px
    # Each stored as (position, span_start, span_end) for clustering.
    h_lines = []
    v_lines = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        if abs(y1 - y2) < 10:
            h_lines.append((min(y1, y2), x1, x2))
        elif abs(x1 - x2) < 10:
            v_lines.append((min(x1, x2), y1, y2))


    # ── Step 5.1: Cluster nearby parallel segments into one line ──
    # Multiple Hough segments may map to the same physical grid line.
    # Groups segments within `threshold` pixels and replaces each
    # group with one representative line (mean position, full span).
    def cluster_lines(lines, axis=0, threshold=15):
        lines.sort()
        merged = []
        group = [lines[0]]
        for line in lines[1:]:
            if abs(line[axis] - group[-1][axis]) <= threshold:
                group.append(line)
            else:
                pos = int(np.mean([l[axis] for l in group]))
                span_min = int(np.min([l[1] for l in group]))
                span_max = int(np.max([l[2] for l in group]))
                merged.append((pos, span_min, span_max))
                group = [line]
        pos = int(np.mean([l[axis] for l in group]))
        span_min = int(np.min([l[1] for l in group]))
        span_max = int(np.max([l[2] for l in group]))
        merged.append((pos, span_min, span_max))
        return merged


    # Apply clustering with 10-pixel tolerance for both directions
    merged_h = cluster_lines(h_lines, axis=0, threshold=10)
    merged_v = cluster_lines(v_lines, axis=0, threshold=10)


    # ── Step 6.1: Retain only full-span line positions ────────────
    # Lines spanning > 3500 px (H) or > 2000 px (V) are accepted
    # as true grid boundaries; shorter detections are discarded.
    filtered_h = [y for y, x1, x2 in merged_h if abs(x2 - x1) > 3500]
    filtered_v = [x for x, y1, y2 in merged_v if abs(y2 - y1) > 2000]


    # ── Step 6.2: Guarantee image-edge boundary lines are present ─
    # Grid detection may miss outermost edges if flush with the
    # image boundary. Adding y=0, y=height-1, x=0, x=width-1
    # ensures every perimeter cell is correctly enclosed.
    if len(filtered_h) < 8:
        if not any(abs(y - 0) < 10 for y in filtered_h): filtered_h.append(0)
        if not any(abs(y - (height - 1)) < 10 for y in filtered_h): filtered_h.append(height - 1)

    if len(filtered_v) < 11:
        if not any(abs(x - 0) < 10 for x in filtered_v): filtered_v.append(0)
        if not any(abs(x - (width - 1)) < 10 for x in filtered_v): filtered_v.append(width - 1)

    # Remove duplicates from edge additions and sort ascending
    filtered_h = sorted(list(set(filtered_h)))
    filtered_v = sorted(list(set(filtered_v)))

    # Optional debug
    print(f"Final Horizontal Lines: {len(filtered_h)} → {filtered_h}")
    print(f"Final Vertical Lines:   {len(filtered_v)} → {filtered_v}")

    # Step 6.2: Draw fully extended valid green lines
    clean_grid = img.copy()
    for y in filtered_h:
        cv2.line(clean_grid, (0, y), (width, y), (0, 255, 0), 2)
    for x in filtered_v:
        cv2.line(clean_grid, (x, 0), (x, height), (0, 255, 0), 2)



    # ── Step 6.3: Extract rectangular cells from grid intersections
    # Each adjacent H-line pair (rows i, i+1) combined with each
    # adjacent V-line pair (cols j, j+1) defines one grid cell
    # containing a single aggregate specimen.
    boxes = []
    for i in range(len(filtered_h) - 1):
        for j in range(len(filtered_v) - 1):
            top = filtered_h[i]
            bottom = filtered_h[i + 1]
            left = filtered_v[j]
            right = filtered_v[j + 1]
            boxes.append(((left, top), (right, bottom)))
    # Step 6.3.1: Calculate and print box areas
    print("\nBox Areas (in pixels):")
    for idx, ((x1, y1), (x2, y2)) in enumerate(boxes, 1):
        width_box = abs(x2 - x1)
        height_box = abs(y2 - y1)
        area = width_box * height_box
        print(f"Box {idx:2d}: Area = {area} px² (Width={width_box}, Height={height_box})")

    # Step 6.4: Draw rectangles, number them, and show area
    for idx, ((x1, y1), (x2, y2)) in enumerate(boxes, 1):
        cv2.rectangle(clean_grid, (x1, y1), (x2, y2), (255, 0, 0), 2)

        # Draw box number
        cv2.putText(clean_grid, str(idx), (x1 + 10, y1 + 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        # Calculate and draw area
        area = abs((x2 - x1) * (y2 - y1))
        cv2.putText(clean_grid, f"A={area}", (x1 + 10, y1 + 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)


    print(f"Total boxes numbered: {len(boxes)}")

    # Step 7: Show final result
    plt.figure(figsize=(12, 12))
    plt.title("Numbered Boxes from Grid Intersections")
    plt.imshow(cv2.cvtColor(clean_grid, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()


    # ── Step 6.5: Select the 70 largest cells by pixel area ───────
    # Partial cells at sheet edges are smaller than interior cells.
    # Ranking by area and selecting top 70 retains only complete
    # aggregate specimens, discarding border fragments.
    box_areas = []
    for ((x1, y1), (x2, y2)) in boxes:
        area = abs((x2 - x1) * (y2 - y1))
        box_areas.append((area, (x1, y1, x2, y2)))

    # Sort descending by area and keep top 70 full-size cells
    box_areas_sorted = sorted(box_areas, key=lambda x: x[0], reverse=True)[:70]

    # Step 6.6: Crop those 70 boxes from original image
    cropped_boxes = []
    for i, (_, (x1, y1, x2, y2)) in enumerate(box_areas_sorted, 1):
        crop = img[y1:y2, x1:x2]
        cropped_boxes.append(crop)

        # Save each crop if needed (optional)
        # filename = f"crop_{i:02d}.png"
        # cv2.imwrite(filename, crop)

    # Step 7: Show first few cropped boxes
    import matplotlib.pyplot as plt

    cols = 5
    rows = min(14, len(cropped_boxes))  # 14x5 = 70 max
    plt.figure(figsize=(15, rows * 2))

    for i, crop in enumerate(cropped_boxes[:70]):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        plt.title(f"Box {i+1}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

    # ── Step 6.6 (refined): Center-crop to fixed 380 x 410 px ───
    # Fixed-size center cropping ensures uniform input dimensions
    # regardless of grid size variation between scanned sheets.
    # The window anchors at cell midpoint (cx, cy) and clamps to
    # image boundaries to prevent out-of-bounds indexing.
    # Centrally crop 380x410 from each of the top 70 boxes
    target_w, target_h = 380, 410
    cropped_boxes = []
    for i, (_, (x1, y1, x2, y2)) in enumerate(box_areas_sorted, 1):
        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2
        x_start = max(0, cx - target_w // 2)
        y_start = max(0, cy - target_h // 2)
        x_end = min(width, x_start + target_w)
        y_end = min(height, y_start + target_h)
        x_start = max(0, x_end - target_w)
        y_start = max(0, y_end - target_h)
        crop = img[y_start:y_end, x_start:x_end]
        cropped_boxes.append(crop)

        # Print size
        h, w = crop.shape[:2]
        print(f"Crop {i:2d}: Width = {w}, Height = {h}")


        # Persist the center-cropped specimen image to Google Drive
        # Save crop to output folder
        filename = f"{os.path.splitext(os.path.basename(img_path))[0]}_{i:02d}.png"
        save_path = os.path.join(output_folder, filename)
        cv2.imwrite(save_path, crop)


# Final confirmation: all specimens have been cropped and saved
print("All images processed and saved to:", output_folder)


Output hidden; open in https://colab.research.google.com to view.